In [ ]:
from pathlib import Path
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings

docs_path = Path("../data/docs")
files = list(docs_path.glob("*.txt"))
print("files:", [f.name for f in files])

docs = []
for p in files:
    docs.extend(TextLoader(str(p), encoding="utf-8").load())

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(docs)[:6]   # cap to keep it fast
print("chunks:", len(chunks))

embeddings = OllamaEmbeddings(model="nomic-embed-text")  # fast embeddings
vs = FAISS.from_documents(chunks, embeddings)

"vs ready"

In [ ]:
from pathlib import Path

Path("../outputs").mkdir(exist_ok=True)
vs.save_local("../outputs/faiss_index")
"saved index"

In [ ]:
from pathlib import Path
Path("../outputs").mkdir(exist_ok=True)
vs.save_local("../outputs/faiss_index")
"saved index"

In [ ]:
import sys
sys.path.append("..")

from src.llm import get_llm
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = get_llm(model="llama3.2:3b")
retriever = vs.as_retriever(search_kwargs={"k": 3})

prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using ONLY the provided context. If missing, say: 'I don't know based on the context.'"),
    ("user", "Question: {question}\n\nContext:\n{context}")
])

def rag_answer(question: str) -> str:
    retrieved = retriever.invoke(question)
    context = "\n\n".join([f"- {d.page_content}" for d in retrieved])
    return (prompt | llm | StrOutputParser()).invoke({"question": question, "context": context})

rag_answer("What databases are mentioned in the docs?")

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="nomic-embed-text")
vs = FAISS.load_local("../outputs/faiss_index", embeddings, allow_dangerous_deserialization=True)

retriever = vs.as_retriever(search_kwargs={"k": 2})
"loaded index"

In [ ]:
import sys
sys.path.append("..")

from src.llm import get_llm
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = get_llm(model="llama3.2:3b")

prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using ONLY the provided context. If missing, say: 'I don't know based on the context.'"),
    ("user", "Question: {question}\n\nContext:\n{context}")
])

def rag_answer_with_citations(question: str):
    retrieved = retriever.invoke(question)

    context = "\n\n".join([f"- {d.page_content[:400]}" for d in retrieved])
    answer = (prompt | llm | StrOutputParser()).invoke({"question": question, "context": context})

    citations = []
    for d in retrieved:
        src = d.metadata.get("source", "unknown")
        snippet = d.page_content[:160].replace("\n", " ")
        citations.append({"source": src, "snippet": snippet})

    return {"answer": answer, "citations": citations}

rag_answer_with_citations("What databases are mentioned in the docs?")

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="nomic-embed-text")
vs = FAISS.load_local("../outputs/faiss_index", embeddings, allow_dangerous_deserialization=True)

retriever = vs.as_retriever(search_kwargs={"k": 2})
"loaded index"

In [ ]:
import sys
sys.path.append("..")

from src.llm import get_llm
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = get_llm(model="llama3.2:3b")

prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using ONLY the provided context. If missing, say: 'I don't know based on the context.'"),
    ("user", "Question: {question}\n\nContext:\n{context}")
])

def rag_answer_with_citations(question: str):
    retrieved = retriever.invoke(question)

    context = "\n\n".join([f"- {d.page_content[:400]}" for d in retrieved])
    answer = (prompt | llm | StrOutputParser()).invoke({"question": question, "context": context})

    citations = []
    for d in retrieved:
        src = d.metadata.get("source", "unknown")
        citations.append({"source": src, "snippet": d.page_content[:160].replace("\n", " ")})

    return {"answer": answer, "citations": citations}

rag_answer_with_citations("What databases are mentioned in the docs?")